## This script assigns stations from station shapefile to HydroBasins (subbasins level 12):

### input files: 
* from BasinATLAS_v10.gdb feature class BasinATLAS_v10_lev12.shp 
* station.shp file 
* stations_assigned_rivers_geodesic.csv
### output files:
*  output table is saved as csv, shp and parquet-file in output_data/aggregate_by_subbasin/stations_info_HYBAS_HYRIV.csv/.shp/.parquet

In [4]:
# import required modules: 
import geopandas as gpd
import fiona
import time
import dask_geopandas
import pyogrio
from scipy.spatial import cKDTree
import pandas as pd
import numpy as np
import os


In [5]:
# define the output folder path:

output_folder = '../../output_data/assign_stations_HydroBasins'

# Check if the folder exists, if not, create it:
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

## 1. read in HydroBasins level 12 feature class:

In [2]:
gdb_path = f'../../input_data/geodata/HydroBasins/BasinATLAS_v10.gdb'


# name of feature-class within geodatabase
feature_class_name = 'BasinATLAS_v10_lev12'

# create complete path to Feature-Class
feature_class_path = f'{gdb_path}\\{feature_class_name}'



In [3]:
# use dask_geopandas to read in feature class of BasinATLAS_v10_lev12 

start = time.time()
layers = fiona.listlayers(gdb_path)
layers
selected_layer = layers[11]

# read in layer 'BasinATLAS_v10_lev12'
#subbasins_lev12 = gpd.GeoDataFrame.from_file(gdb_path,layer= 'BasinATLAS_v10_lev12')
data_level12 = dask_geopandas.read_file(gdb_path,layer= 'BasinATLAS_v10_lev12', npartitions = 4)
end = time.time()

runtime = (end - start)
print(runtime)

0.4629840850830078


In [4]:
# now put partitions together again: to obtain one large df:

start = time.time()
subbasins_lev12 = data_level12.compute()
end= time.time()
dauer = (end-start)
print(f'{dauer}sec')

198.48052287101746sec


## 2. read in station.shp file: 

In [5]:
#  read in stations shp file:

stations = gpd.read_file('../../output_data/merged_datasets/stations.shp')
# add FID column: by index 
stations['FID'] = stations.index
stations[1:10]

,dataset,site_id,data_sourc,country,area,lat,lon,geometry,FID
1,arcticdeltas,Yenisey,ArcticGRO,Russia,2500000.0,69.380000,86.150000,POINT (86.15000 69.38000),1
2,arcticdeltas,Lena,ArcticGRO,Russia,2400000.0,66.770000,123.370000,POINT (123.37000 66.77000),2
3,arcticdeltas,Kolyma,ArcticGRO,Russia,1800000.0,68.750000,161.300000,POINT (161.30000 68.75000),3
4,arcticdeltas,Yukon,ArcticGRO,USA,830000.0,61.930000,-162.880000,POINT (-162.88000 61.93000),4
5,arcticdeltas,Mackenzie,ArcticGRO,Canada,650000.0,67.450000,-133.740000,POINT (-133.74000 67.45000),5
6,denmark,1000039,Overfladevandsdatabasen,Denmark,-9999.0,57.035843,8.567386,POINT (8.56739 57.03584),6
7,denmark,1000040,Overfladevandsdatabasen,Denmark,-9999.0,57.047458,8.515677,POINT (8.51568 57.04746),7
8,denmark,1000091,Overfladevandsdatabasen,Denmark,-9999.0,57.033169,8.521300,POINT (8.52130 57.03317),8
9,denmark,1000102,Overfladevandsdatabasen,Denmark,-9999.0,57.360521,9.780605,POINT (9.78061 57.36052),9


In [6]:
# convert HYBAS_ID to string:

subbasins_lev12['HYBAS_ID'] = subbasins_lev12['HYBAS_ID'].astype(np.int64).astype(str)
subbasins_lev12['HYBAS_ID'].max()

'9120169950'

## 3. Now assign stations to polygons of HydroBasins level 12: apply geopandas spatialjoin function:

In [7]:
 # spatial join: 
result = gpd.sjoin(stations, subbasins_lev12, how='left', predicate='within')

In [8]:
result[result['HYBAS_ID'].isna()]

,dataset,site_id,data_sourc,country,area,lat,lon,geometry,FID,index_right,...,hft_ix_u93,hft_ix_s09,hft_ix_u09,gad_id_smj,gdp_ud_sav,gdp_ud_ssu,gdp_ud_usu,hdi_ix_sav,Shape_Length,Shape_Area
169,denmark,9000799,Overfladevandsdatabasen,Denmark,-9999.0,57.071254,9.651265,POINT (9.65127 57.07125),169,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7767,germany,SH_120207,GDS2,Germany,-9999.0,53.879512,9.168620,POINT (9.16862 53.87951),7767,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8093,germany,NI_59752051,GDS2,Germany,-9999.0,53.677877,9.495274,POINT (9.49527 53.67788),8093,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14362,GRQA,BRA00173,GEMSTAT,Brazil,-9999.0,-0.169861,-50.537889,POINT (-50.53789 -0.16986),14362,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14364,GRQA,BRA00175,GEMSTAT,Brazil,-9999.0,-0.065694,-51.124306,POINT (-51.12431 -0.06569),14364,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70528,GRQA,USGS-221243159283900,WQP,United States,-9999.0,22.208803,-159.474683,POINT (-159.47468 22.20880),70528,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70529,GRQA,USGS-221247159285400,WQP,United States,-9999.0,22.209914,-159.478850,POINT (-159.47885 22.20991),70529,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70548,GRQA,USGS-292916090005800,WQP,United States,-9999.0,29.487996,-90.016184,POINT (-90.01618 29.48800),70548,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
73496,USGS,dcPMS37,DC DDOE,USA,30768.8,38.821780,-77.031090,POINT (-77.03109 38.82178),73496,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# select required columns: 
station_basins = result[['dataset', 'site_id', 'data_sourc', 'country', 'area', 'lat', 'lon', 'geometry', 'FID', 'HYBAS_ID']]
station_basins['HYBAS_ID'] = station_basins['HYBAS_ID']#.astype(np.int64)
station_basins[0:10]

C:\Users\bartusch\cnp_env\lib\site-packages\geopandas\geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,dataset,site_id,data_sourc,country,area,lat,lon,geometry,FID,HYBAS_ID
0,arcticdeltas,Ob,ArcticGRO,Russia,2950000.0,66.630000,66.600000,POINT (66.60000 66.63000),0,3120792150
1,arcticdeltas,Yenisey,ArcticGRO,Russia,2500000.0,69.380000,86.150000,POINT (86.15000 69.38000),1,3120110580
2,arcticdeltas,Lena,ArcticGRO,Russia,2400000.0,66.770000,123.370000,POINT (123.37000 66.77000),2,3120183000
3,arcticdeltas,Kolyma,ArcticGRO,Russia,1800000.0,68.750000,161.300000,POINT (161.30000 68.75000),3,3120126160
4,arcticdeltas,Yukon,ArcticGRO,USA,830000.0,61.930000,-162.880000,POINT (-162.88000 61.93000),4,8120251750
5,arcticdeltas,Mackenzie,ArcticGRO,Canada,650000.0,67.450000,-133.740000,POINT (-133.74000 67.45000),5,8120122010
6,denmark,1000039,Overfladevandsdatabasen,Denmark,-9999.0,57.035843,8.567386,POINT (8.56739 57.03584),6,2120024920
7,denmark,1000040,Overfladevandsdatabasen,Denmark,-9999.0,57.047458,8.515677,POINT (8.51568 57.04746),7,2120024920
8,denmark,1000091,Overfladevandsdatabasen,Denmark,-9999.0,57.033169,8.521300,POINT (8.52130 57.03317),8,2120024920
9,denmark,1000102,Overfladevandsdatabasen,Denmark,-9999.0,57.360521,9.780605,POINT (9.78061 57.36052),9,2120024940


## 4. Now add info about closest and second closest HydroRiverID to stations_basins df. Use arcpy generated stations_assigned_rivers_geodesic.csv file:

In [10]:
# read in stations_assigned_rivers01.csv:
table_closest_river_id = pd.read_csv('../../output_data/assign_stations_HydroRivers/stations_assigned_rivers_geodesic.csv', sep = ';', decimal = ',')

table_closest_river_id['OBJECTID'] = table_closest_river_id.index


# some stations have same distance to more than one river segment --> adapt rank in duplicated row and flag in another column
condition_dup_seg =table_closest_river_id[(table_closest_river_id.duplicated(subset = ['IN_FID'], keep = 'first')) & (table_closest_river_id['NEAR_RANK']==1)]
table_closest_river_id2 = table_closest_river_id.copy()
table_closest_river_id2.loc[condition_dup_seg.index, 'NEAR_RANK'] = 2

# flag rows where second closest river id has the same distance as the first one: flag 0 distance is not equal; 1 distance to closest and second closest river segment is equal
table_closest_river_id2['flag_rivID'] = 0
table_closest_river_id2.loc[condition_dup_seg.index, 'flag_rivID'] = 1



## read in HydroRiver shapefile:

In [11]:
# read in HydroRivers.shp file: 
hydroriver_shp_dask = dask_geopandas.read_file('../../input_data/geodata/HydroRIVERS/HydroRIVERS_v10_shp/HydroRIVERS_v10.shp', npartitions = 4)
hydroriver_shp = hydroriver_shp_dask.compute()
hydroriver_shp['FID'] = hydroriver_shp.index


## add original HydroRivers ID: 
* the information is added based on NEAR_FID (index of HydroRivers.shp file)

In [16]:

# IN_FID describes row number in stations.shp (FID column in stations)
# NEAR_FID describes row number in HydroRivers.shp
closest_river_id = table_closest_river_id2.copy()
closest_river_correct_id = closest_river_id.merge(hydroriver_shp[['HYRIV_ID','ORD_STRA', 'HYBAS_L12', 'FID']], how = 'left',
                left_on = 'NEAR_FID', right_on = 'FID').drop(columns = ['FID'])


In [17]:
closest_river_correct_id[0:100]

,OBJECTID,IN_FID,NEAR_FID,NEAR_DIST,NEAR_RANK,flag_rivID,HYRIV_ID,ORD_STRA,HYBAS_L12
0,0,0,2615579,1068.797743,1,0,30149485,9,3120792150
1,1,0,2615580,1103.942531,2,0,30149486,1,3120792150
2,2,1,2541407,1112.490196,1,0,30075313,1,3120110580
3,3,1,2541197,1359.622757,2,0,30075103,9,3120110580
4,4,2,2612207,682.567840,1,0,30146113,3,3120183000
...,...,...,...,...,...,...,...,...,...
95,95,47,1731968,103.684817,2,0,20204418,3,2120024970
96,96,48,1734052,55.264702,1,0,20206502,3,2121072510
97,97,48,1735022,2342.706086,2,0,20207472,1,2121072510
98,98,49,1733911,1025.033574,1,0,20206361,1,2120024970


## split dataframe closest river correct id by rank 1 and 2 and merge info about nearest and 2ndnearest river id to stations_basin dataframe:

In [18]:
# get all first rank river ids: 
cl_river_r1 = closest_river_correct_id[closest_river_correct_id['NEAR_RANK']==1]

# rename columns by adding prefix to columns: 1
cl_river_r1_a = cl_river_r1.add_prefix('1st').drop(['1stOBJECTID', '1stNEAR_RANK', '1stflag_rivID'], axis=1) # drop 1stOBJECTID, 1stNEAR_RANK, 1stflag_rivID

# get all second rank river ids:
cl_river_r2 = closest_river_correct_id[closest_river_correct_id['NEAR_RANK']==2]
cl_river_r2_a = cl_river_r2.add_prefix('2nd').drop(['2ndOBJECTID','2ndNEAR_RANK'], axis = 1).rename(columns = {'2ndflag_rivID':'flag_samerivID'})

# merge dataframes

stations_info_basin_river1 = pd.merge(station_basins, cl_river_r1_a, how='inner', left_on='FID', right_on='1stIN_FID').drop(columns = ['1stIN_FID'])
stations_info_basin_river2 = pd.merge(stations_info_basin_river1, cl_river_r2_a, how='inner', left_on='FID', right_on='2ndIN_FID').drop(columns = ['2ndIN_FID', '2ndHYBAS_L12'])                                                                                                                   

#### Add information about min , max and median Strahler Order in hydrobasins

In [19]:
# compute  maximum and minimum stream order in subbasins 
stream_order_subbasins = hydroriver_shp.groupby('HYBAS_L12')['ORD_STRA'].agg(['max','min', 'median', 'count']).rename(columns = {'max':'max_str_ord', 'min':'min_str_ord', 'median':'median_str_ord', 'count':'n_river_seg_basin'})
stream_order_subbasins = stream_order_subbasins.reset_index()
# merge to 
stream_order_subbasins['HYBAS_L12'] = stream_order_subbasins['HYBAS_L12'].astype(np.int64).astype(str)
stations_info_basin_river3 = pd.merge(stations_info_basin_river2, stream_order_subbasins, how='left', left_on='HYBAS_ID', right_on='HYBAS_L12').drop(columns = ['HYBAS_L12'])



# rename columns 1stNEAR_DIST and  2ndNEAR_DIST by adding m for distance unit in meters: 
stations_info_basin_river3 = stations_info_basin_river3.rename(columns = {'1stNEAR_DIST':'1stNEAR_DIST_m', '2ndNEAR_DIST':'2ndNEAR_DIST_m'})




In [ ]:
stations_info_basin_river3[0:10]

,dataset,site_id,data_sourc,country,area,lat,lon,geometry,FID,HYBAS_ID,...,1stHYBAS_L12,2ndNEAR_FID,2ndNEAR_DIST_m,flag_samerivID,2ndHYRIV_ID,2ndORD_STRA,max_str_ord,min_str_ord,median_str_ord,n_river_seg_basin
0,arcticdeltas,Ob,ArcticGRO,Russia,2950000.0,66.630000,66.600000,POINT (66.60000 66.63000),0,3120792150,...,3120792150,2615580,1103.942531,0,30149486,1,9.0,1.0,5.0,4.0
1,arcticdeltas,Yenisey,ArcticGRO,Russia,2500000.0,69.380000,86.150000,POINT (86.15000 69.38000),1,3120110580,...,3120110580,2541197,1359.622757,0,30075103,9,9.0,1.0,2.0,19.0
2,arcticdeltas,Lena,ArcticGRO,Russia,2400000.0,66.770000,123.370000,POINT (123.37000 66.77000),2,3120183000,...,3120183000,2611503,687.243313,0,30145409,1,3.0,1.0,3.0,5.0
3,arcticdeltas,Kolyma,ArcticGRO,Russia,1800000.0,68.750000,161.300000,POINT (161.30000 68.75000),3,3120126160,...,3120126160,2556389,343.466012,0,30090295,8,8.0,1.0,2.0,7.0
4,arcticdeltas,Yukon,ArcticGRO,USA,830000.0,61.930000,-162.880000,POINT (-162.88000 61.93000),4,8120251750,...,8120251750,8219711,1009.868410,0,80202995,1,8.0,1.0,2.0,19.0
5,arcticdeltas,Mackenzie,ArcticGRO,Canada,650000.0,67.450000,-133.740000,POINT (-133.74000 67.45000),5,8120122010,...,8120122010,8091448,517.314803,0,80074732,6,9.0,1.0,9.0,9.0
6,denmark,1000039,Overfladevandsdatabasen,Denmark,-9999.0,57.035843,8.567386,POINT (8.56739 57.03584),6,2120024920,...,2120024920,1741477,1600.648089,0,20213927,1,2.0,1.0,1.0,11.0
7,denmark,1000040,Overfladevandsdatabasen,Denmark,-9999.0,57.047458,8.515677,POINT (8.51568 57.04746),7,2120024920,...,2120024920,1742140,572.159002,0,20214590,1,2.0,1.0,1.0,11.0
8,denmark,1000091,Overfladevandsdatabasen,Denmark,-9999.0,57.033169,8.521300,POINT (8.52130 57.03317),8,2120024920,...,2120024920,1741476,1324.103373,0,20213926,2,2.0,1.0,1.0,11.0
9,denmark,1000102,Overfladevandsdatabasen,Denmark,-9999.0,57.360521,9.780605,POINT (9.78061 57.36052),9,2120024940,...,2120024940,1736431,3722.274865,0,20208881,2,2.0,1.0,1.0,5.0


## save data as shapefile and csv file

In [21]:

# save as csv-file: 
stations_info_basin_river3.to_csv('../../output_data/assign_stations_HydroBasins/stations_info_HYBAS_HYRIV.csv')

# save as shp-file: 
gdf = gpd.GeoDataFrame(stations_info_basin_river3, crs="EPSG:4326")
gdf = gdf.set_index(['dataset','site_id'])

gdf.to_file('../../output_data/assign_stations_HydroBasins/stations_info_HYBAS_HYRIV.shp')
# and additional to parquet file: 
gdf.to_parquet('../../output_data/assign_stations_HydroBasins/stations_info_HYBAS_HYRIV.parquet')

C:\Users\bartusch\AppData\Local\Temp\8\ipykernel_30392\1794078137.py:8: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file('../../output_data/assign_stations_HydroBasins/stations_info_HYBAS_HYRIV.shp')


### save info about streams and stream orders in csv file:

In [22]:
stream_order_subbasins.to_csv('../../output_data/assign_stations_HydroBasins/info_HYBAS_strahler.csv')